In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/FlowTransformer_Pytorch_Imp

/content/drive/MyDrive/FlowTransformer_Pytorch_Imp


In [4]:
import os

import pandas as pd
import pdb
from framework.dataset_specification import NamedDatasetSpecifications
from framework.enumerations import EvaluationDatasetSampling
from framework.flow_transformer import FlowTransformer
from framework.flow_transformer_parameters import FlowTransformerParameters
from framework.framework_component import FunctionalComponent
from implementations.classification_heads import *
from implementations.input_encodings import *
from implementations.pre_processings import StandardPreProcessing
from implementations.transformers.basic_transformers import BasicTransformer
from implementations.transformers.named_transformers import *

encodings = [
    NoInputEncoder(),
    RecordLevelEmbed(64),
    CategoricalFeatureEmbed(EmbedLayerType.Dense, 16),
    CategoricalFeatureEmbed(EmbedLayerType.Lookup, 16),
    CategoricalFeatureEmbed(EmbedLayerType.Projection, 16),
    RecordLevelEmbed(64, project=True)
]

classification_heads = [
    LastTokenClassificationHead(),
    FlattenClassificationHead(),
    GlobalAveragePoolingClassificationHead(),
    CLSTokenClassificationHead(),
    FeaturewiseEmbedding(project=False),
    FeaturewiseEmbedding(project=True),
]

transformers: List[FunctionalComponent] = [
    BasicTransformer(2, 128, n_heads=2),
    BasicTransformer(2, 128, n_heads=2, is_decoder=True),
    GPTSmallTransformer(),
    BERTSmallTransformer()
]

flow_file_path = r"C:\Data\UQ\NIDS\Collected"
# os.path.join(flow_file_path, "NF-CSE-CIC-IDS2018-v2.csv")
datasets = [
    ("firewall_test_20250731_160700_flows", '/content/drive/MyDrive/FlowTransformer_Pytorch_Imp/firewall_test_20250731_160700-flows.csv', NamedDatasetSpecifications.my_data, 0.01, EvaluationDatasetSampling.LastRows)
]

pre_processing = StandardPreProcessing(n_categorical_levels=32)

# Define the transformer
ft = FlowTransformer(pre_processing=pre_processing,
            input_encoding=encodings[5],
            sequential_model=transformers[0],
            classification_head=classification_heads[0],
            params=FlowTransformerParameters(window_size=32, mlp_layer_sizes=[128], mlp_dropout=0.1))

# Load the specific dataset
dataset_name, dataset_path, dataset_specification, eval_percent, eval_method = datasets[0]
# pdb.set_trace()
ft.load_dataset(dataset_name, dataset_path, dataset_specification, evaluation_dataset_sampling=eval_method, evaluation_percent=eval_percent)

# Build the transformer model
m = ft.build_model()
m.summary()

# Compile the model
# m.compile(optimizer="adam", loss='binary_crossentropy', metrics=['binary_accuracy'], jit_compile=True)
m.compile(optimizer="adam", loss='binary_crossentropy', metrics=['binary_accuracy'])

# Get the evaluation results
eval_results: pd.DataFrame
(train_results, eval_results, final_epoch) = ft.evaluate(m, batch_size=128, epochs=500, steps_per_epoch=64, early_stopping_patience=20)


print(eval_results)

Using cache file path: cache/firewall_test_20250731_160700_flows_0_QdLmZHuh8yOmlGcKBEkf7hepImY0_VHNk9ujbqtTXGSrgVayeqG486IQ0.feather
Reading directly from cache cache/firewall_test_20250731_160700_flows_0_QdLmZHuh8yOmlGcKBEkf7hepImY0_VHNk9ujbqtTXGSrgVayeqG486IQ0.feather...


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_flow_iat_min  │ (None, 32, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_fwd_iat_std   │ (None, 32, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_max_packet_l… │ (None, 32, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_fwd_header_l… │ (None, 32, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_idle_max      │ (None, 32, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_bwd_packet_l… │ (None, 32, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_min_packet_l… │ (None, 32, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_psh_flag_cou… │ (None, 32, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_flow_packets… │ (None, 32, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_average_pack… │ (None, 32, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_active_mean   │ (None, 32, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_total_fwd_pa… │ (None, 32, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_idle_std      │ (None, 32, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_active_max    │ (None, 32, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_flow_iat_mean │ (None, 32, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_avg_bwd_segm… │ (None, 32, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_packet_lengt… │ (None, 32, 1)     │          0 │ -               

 Total params: 183,809 (718.00 KB)

 Trainable params: 183,809 (718.00 KB)

 Non-trainable params: 0 (0.00 B)

Building eval dataset...
Splitting dataset to featurewise...
Evaluation dataset is built!
Positive samples in eval set: 381
Negative samples in eval set: 5
Epoch = 0 / 500 (early stop in 20), step = 0, loss = 0.80497, results = [array(0.8049675, dtype=float32), array(0.4921875, dtype=float32)] -- elapsed (train): 0.00s
Epoch = 0 / 500 (early stop in 20), step = 36, loss = 0.36859, results = [array(0.36859006, dtype=float32), array(0.8391047, dtype=float32)] -- elapsed (train): 1.55s
Epoch = 1 / 500 (early stop in 20), step = 22, loss = 0.32751, results = [array(0.327514, dtype=float32), array(0.86332613, dtype=float32)] -- elapsed (train): 3.32s
Epoch = 2 / 500 (early stop in 20), step = 9, loss = 0.30856, results = [array(0.30856174, dtype=float32), array(0.87375456, dtype=float32)] -- elapsed (train): 5.07s
Epoch = 2 / 500 (early stop in 20), step = 60, loss = 0.29242, results = [array(0.2924242, dtype=float32), array(0.88074577, dtype=float32)] -- elapsed (train): 6.86s
Epoch = 3 / 